In [1]:
# 04_pareto_frontier.ipynb -- Combine utility and residual risk into a Pareto frontier and overlay the regulatory (non-processing) threshold.
# Pulls per-tier metrics produced by notebooks 02 and 03; produces the main frontier figure and a compliance-vs-utility figure.
# Output paths are intentionally not printed (anonymity).

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Reproducibility & grayscale publication style -------------------------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
    "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys",
})

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

# --- Merge per-tier metrics ------------------------------------------------
recall = pd.read_csv(TAB / "recall_by_tier.csv")
util = pd.read_csv(TAB / "utility_by_tier.csv")
risk = pd.read_csv(TAB / "risk_by_tier.csv")
df = recall.merge(util, on="tier").merge(risk, on="tier")

# Regulatory "non-processing" threshold: residual regulated-PII leakage must not exceed THRESHOLD.
THRESHOLD = 0.0
df["compliant"] = df["leakage"] <= THRESHOLD
df.to_csv(TAB / "pareto_summary.csv", index=False)

# Manual label offsets (points coincide where tiers share retention; e.g., C0=C1 and C3=C4).
OFFSETS = {"C0": (9, 8), "C1": (9, -15), "C2": (10, 6), "C3": (-26, 9), "C4": (9, -15)}

# --- Figure: Pareto frontier (utility vs reconstruction risk) --------------
# Filled marker = regulatory-compliant tier; open marker = non-compliant (PII leakage above threshold).
fig, ax = plt.subplots(figsize=(5.4, 4.0))
has_ci = "reconstruction_risk_lo" in df.columns
for _, row in df.iterrows():
    face = "0.25" if row["compliant"] else "white"
    if has_ci:  # horizontal 95% CI on the measured reconstruction risk
        xe = [[max(0.0, row["reconstruction_risk"] - row["reconstruction_risk_lo"])],
              [max(0.0, row["reconstruction_risk_hi"] - row["reconstruction_risk"])]]
        ax.errorbar(row["reconstruction_risk"], row["task_success_proxy"], xerr=xe,
                    fmt="none", ecolor="0.45", elinewidth=1.0, capsize=2, zorder=2)
    ax.scatter(row["reconstruction_risk"], row["task_success_proxy"], s=75,
               facecolors=face, edgecolors="0.1", linewidths=1.3, zorder=3)
    dx, dy = OFFSETS[row["tier"]]
    ax.annotate(row["tier"], (row["reconstruction_risk"], row["task_success_proxy"]),
                textcoords="offset points", xytext=(dx, dy), fontsize=10)

# Pareto frontier over compliant tiers (non-dominated: lower risk, higher utility).
comp = df[df["compliant"]].sort_values("reconstruction_risk")
frontier, best_u = [], -1.0
for _, r in comp.iterrows():
    if r["task_success_proxy"] >= best_u:
        frontier.append((r["reconstruction_risk"], r["task_success_proxy"]))
        best_u = r["task_success_proxy"]
if len(frontier) >= 2:
    fx, fy = zip(*frontier)
    ax.plot(fx, fy, color="0.1", linewidth=1.4, zorder=2)
ax.set_xlabel("Reconstruction risk")
ax.set_ylabel("Utility (task-success proxy)")
ax.set_xlim(-0.05, 1.03)
ax.set_ylim(-0.03, 1.03)
save_fig(fig, "fig_pareto_frontier")

# --- Figure: compliance vs utility with regulatory threshold line ----------
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for _, row in df.iterrows():
    face = "0.25" if row["compliant"] else "white"
    ax.scatter(row["leakage"], row["task_success_proxy"], s=75,
               facecolors=face, edgecolors="0.1", linewidths=1.3, zorder=3)
    cdx, cdy = {"C0": (-26, 6), "C1": (9, 6), "C2": (9, 6), "C3": (9, 7), "C4": (9, -16)}[row["tier"]]
    ax.annotate(row["tier"], (row["leakage"], row["task_success_proxy"]),
                textcoords="offset points", xytext=(cdx, cdy), fontsize=10)
ax.axvline(THRESHOLD, color="0.1", linestyle="--", linewidth=1.3, zorder=1)
ax.axvspan(-0.05, THRESHOLD, color="0.90", alpha=0.7, zorder=0)
ax.set_xlabel("Residual regulated-PII leakage (1 - recall)")
ax.set_ylabel("Utility (task-success proxy)")
ax.set_xlim(-0.05, 1.03)
ax.set_ylim(-0.03, 1.03)
save_fig(fig, "fig_compliance_utility")

print("pareto summary:")
print(df[["tier", "leakage", "context_retention", "task_success_proxy",
          "reconstruction_risk", "reversible", "compliant"]].to_string(index=False))


pareto summary:
tier  leakage  context_retention  task_success_proxy  reconstruction_risk  reversible  compliant
  C0      1.0               1.00            0.964429             1.000000        True      False
  C1      0.0               1.00            0.964429             1.000000        True       True
  C2      0.0               0.55            0.645656             0.324719        True       True
  C3      0.0               0.30            0.289050             0.319701        True       True
  C4      0.0               0.30            0.289050             0.319701       False       True
